In [1]:
import pandas as pd
from typing import Optional, cast
from google.cloud import bigquery
from google.cloud.bigquery.job import LoadJob
import json

path = '/Users/jeanbizot/Documents/projets/BIZAPP/tomorro/data/'
ORG = "organizations"
CONTRACTS = "contract_events"
PROJECT_ID = "bzt-ingestion-prod"

bq_client = bigquery.Client(PROJECT_ID)

def df_to_bq(
        df: pd.DataFrame,
        table_id: str,
        client: bigquery.Client,
        write_disposition: str,  # Legacy argument
        labels: Optional[dict[str, str]] = None,
) -> LoadJob:
    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        write_disposition=write_disposition,
        labels={"app": "python", **(labels or {})},
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    return cast(LoadJob, job.result())

def load_json_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    return df

In [2]:
for table in [ORG, CONTRACTS]:
    df = load_json_data(path + table + ".json")
    table_id = PROJECT_ID + ".raw." + table
    df_to_bq(df=df, table_id=table_id, client=bq_client, write_disposition="WRITE_TRUNCATE")